In [ ]:
%pip install -e ../../../

This is for the dynamic properties. We need to set up the costs so that they make sense.
Inflation will do the rest. If we want to reflect a technological advancement or something we can just break
the trend ourselves.

For sources we have:
unit cost by type 
fixed operational rates by type
volumetric operational rates by type 

desalination will have huge values until 2025 and probably later
unit costs will follow inflation but in 2017 a new innovation made unit cost for a new surface plant 

From the Vewin report Chpater 3.3, Figure 3.4 Surface water makes up 40% of total consumption.
Same from Figure 3.7 the highest forecast is 1250, but we are basically already there, so I will use 1300
M m^3 of demand... 

for 2024, the Ferrraa Team has: 63821,72 L/s peak demand, 38861 L/s average
which means 2.01 billions M^3 of or 1.225 

In [ ]:
import itertools
import numpy as np
import pandas as pd

from water_futures_battle.core.base_model import DynamicProperties

static_properties_path = "../sources-static_properties.xlsx"
sources_sps = pd.read_excel(static_properties_path,
                                  sheet_name=None)

Y2000 = pd.Timestamp(year=2000, month=1, day=1)
Y2024 = pd.Timestamp(year=2024, month=1, day=1)

from water_futures_battle.economy.dynamic_properties import EconomyDB
from water_futures_battle.economy.services import adjust_for_inflation

econ_db = EconomyDB.load_from_file('../../economy/economy-dynamic_properties.xlsx')
inflation = econ_db[EconomyDB.INFLATION]['NL0000']
inflation = inflation[inflation.index <= Y2024]

## Dynamic properties and how they change with time

### Unit cost

how much it costs for to build a one cubic meter per day (or whatever will be the unit of the nominal capacity) facility
the price increases with inflation, the price changes deterministically with the source size (bigger sources have a light discount)

### Fixed operational cost

Yearly expenditure per unit of nominal capacity to run that facility 
Unit: €/year
it increases with inflation, it changes with the source size (bigger sources have a light discount), and it is uncertain, we give a min-max bound that will be used to withdraw the actual cost from a uniform distribution

### Volumetric operational cost related to energy

Energy spent for production
Unit: kWh/m^3
it increases with inflation, it changes with the source size (bigger sources have a light discount), and it is uncertain, we give a min-max bound that will be used to withdraw the actual cost from a uniform distribution

### Volumetric operational cost related to non energy related costs

Money spent for production
Unit: €/m^3
it increases with inflation, it changes with the source size (bigger sources have a light discount), and it is uncertain, we give a min-max bound that will be used to withdraw the actual source cost from a uniform distribution

In [ ]:
groundw_dps = DynamicProperties('groundwater-dynamic_properties')
surfacew_dps = DynamicProperties('surface_water-dynamic_properties')
desalin_dps = DynamicProperties('desalination-dynamic_properties')
DPS = [groundw_dps, surfacew_dps, desalin_dps]

LOCATIONS = ['NL0000']
SOURCE_SIZES = ['SMALL', 'MEDIUM', 'LARGE', 'VERY_LARGE']
UNIF_DIST_PARAMS = ['min', 'max']

In [ ]:
UNIT_COSTS_COLUMNS = [f"{c[0]}-{c[1]}" for c in itertools.product(LOCATIONS, SOURCE_SIZES)]

groundw_df = pd.DataFrame(
    data=[[
        600,
        550,
        500,
        450
    ]],
    columns=UNIT_COSTS_COLUMNS,
    index=pd.DatetimeIndex([Y2024], name='timestamp')
)

surfacew_df = pd.DataFrame(
    data=[[
        np.nan,
        np.nan,
        1000,
        900
    ]],
    columns=UNIT_COSTS_COLUMNS,
    index=pd.DatetimeIndex([Y2024], name='timestamp')
)

desalin_df = pd.DataFrame(
    data=[[
        np.nan,
        np.nan,
        1400,
        1200
    ]],
    columns=UNIT_COSTS_COLUMNS,
    index=pd.DatetimeIndex([Y2024], name='timestamp')
)

for dps, df in zip(DPS, [groundw_df,surfacew_df,desalin_df]):
    dps['new_source-unit_cost'] = adjust_for_inflation(
        df.loc[Y2024], Y2024, inflation
    ).loc[[Y2000]].copy()


In [ ]:
DYNCOSTS_COLUMNS = [f"{c[0]}-{c[1]}-{c[2]}" for c in itertools.product(LOCATIONS, SOURCE_SIZES, UNIF_DIST_PARAMS)]

groundw_df = pd.DataFrame(
    data=[[
        0.05, 0.4,
        0.05, 0.2,
        0.05, 0.1,
        0.05, 0.1
    ]],
    columns=DYNCOSTS_COLUMNS,
    index=pd.DatetimeIndex([Y2024], name='timestamp')
)

surfacew_df = pd.DataFrame(
    data=[[
        np.nan, np.nan,
        np.nan, np.nan,
        0.22, 0.24,
        0.19, 0.21
    ]],
    columns=DYNCOSTS_COLUMNS,
    index=pd.DatetimeIndex([Y2024], name='timestamp')
)

desalin_df = pd.DataFrame(
    data=[[
        np.nan, np.nan,
        np.nan, np.nan,
        0.34, 0.36,
        0.30, 0.32
    ]],
    columns=DYNCOSTS_COLUMNS,
    index=pd.DatetimeIndex([Y2024], name='timestamp')
)

for dps, df in zip(DPS, [groundw_df,surfacew_df,desalin_df]):
    dps['opex-fixed'] = adjust_for_inflation(
        df.loc[Y2024], Y2024, inflation
    ).loc[[Y2000]].copy()

In [ ]:
DYNCOSTS_COLUMNS = [f"{c[0]}-{c[1]}-{c[2]}" for c in itertools.product(LOCATIONS, SOURCE_SIZES, UNIF_DIST_PARAMS)]

groundw_df = pd.DataFrame(
    data=[[
        0.05, 0.1,
        0.05, 0.1,
        0.05, 0.1,
        0.05, 0.1
    ]],
    columns=DYNCOSTS_COLUMNS,
    index=pd.DatetimeIndex([Y2024], name='timestamp')
)

surfacew_df = pd.DataFrame(
    data=[[
        np.nan, np.nan,
        np.nan, np.nan,
        0.14, 0.17,
        0.14, 0.16
    ]],
    columns=DYNCOSTS_COLUMNS,
    index=pd.DatetimeIndex([Y2024], name='timestamp')
)

desalin_df= pd.DataFrame(
    data=[[
        np.nan, np.nan,
        np.nan, np.nan,
        0.24, 0.27,
        0.24, 0.26
    ]],
    columns=DYNCOSTS_COLUMNS,
    index=pd.DatetimeIndex([Y2024], name='timestamp')
)

for dps, df in zip(DPS, [groundw_df,surfacew_df,desalin_df]):
    dps['opex-volum-other'] = adjust_for_inflation(
        df.loc[Y2024], Y2024, inflation
    ).loc[[Y2000]].copy()

## Availability factor

Sources may not be always available and their nominal capacity coudl be temporarily reduced.

capacity-effective = capacity-nominal x availability_factor

In [ ]:
import numpy as np
from numpy.random import default_rng
RNG = default_rng(12808)
# Desalination and groundwater, for now have no availability factor variations.
# So we put 1 at the beginning for the national net and it will stay that way
df = pd.DataFrame(data=[[1.0]], columns=LOCATIONS, index=pd.DatetimeIndex(["2000-01-01"], name='timestamp'))
groundw_dps['availability_factor'] = df.copy()
desalin_dps['availability_factor'] = df.copy()

AF_COLUMNS = ['NL0000', 'Rhine', 'Meuse']
# for now just nl0000
df = pd.DataFrame(
    data=[[1.,1.,1.]],
    columns=AF_COLUMNS,
    index=pd.DatetimeIndex(["2000-01-01"], name='timestamp'))

# For surface water, Table 2.4 of the Vewin report reports Intake stops and 
# restrictions at drinking water intake points (days) taledn from the respective RIWA
# For each year we sample on which day it starts.
# However the table is in equivalent days, and especially for the Mesue this is quite a big number
shutoff_eqdays = {
    2011: {'Rhine': 13, 'Meuse': 65},
    2012: {'Rhine': 4, 'Meuse': 114},
    2013: {'Rhine': 15, 'Meuse': 126},
    2014: {'Rhine': 39, 'Meuse': 209},
    2015: {'Rhine': 24, 'Meuse': 478},
    2016: {'Rhine': 6, 'Meuse': 308},
    2017: {'Rhine': 0, 'Meuse': 59},
    2018: {'Rhine': 90, 'Meuse': 46},
    2019: {'Rhine': 37, 'Meuse': 65},
    2020: {'Rhine': 60, 'Meuse': 174}
}

# for each year, select a random date over the year.
# divide by the number of sources (3 for Rhine and 2 for Meuse)
# also, Meuse need to be divided by the MEUSE_CORR_FACTOR 
# the int coming out of this calculation is the number of days at which we have limited capacity

SOURCE_COUNTS = {'Rhine': 3, 'Meuse': 2}
CORR_FACTORS = {'Rhine': 1.0, 'Meuse': 0.7}
MAX_DURATION = 120 # Maximum number of days for the restriction period
for year, eqdays in shutoff_eqdays.items():
    for basin in AF_COLUMNS[1:]:
        eff_days = int(eqdays[basin]/SOURCE_COUNTS[basin]*CORR_FACTORS[basin])

        rand_days = RNG.integers(eff_days, MAX_DURATION+1) if eff_days < MAX_DURATION else eff_days

        af = 1.0 - float(eff_days)/float(rand_days)

        T_start = RNG.choice(
            pd.date_range(start=pd.to_datetime(f'{year}-03-01'),
                          end=pd.to_datetime(f'{year}-05-31'))
        )

        T_end = T_start + pd.Timedelta(days=rand_days)

        df.loc[T_start, basin] = af
        df.loc[T_end, basin] = 1.0

df.sort_index(inplace=True)
surfacew_dps['availability_factor'] = df.ffill()

## Fine for water displacement table 

This doesn't change with inflation but is decided by the governemnt every x years.

Deviations are categorized in MILD, SEVERE, EXTREME. 
For each of these categories you pay a fine which is expressed in €, independently
of the amount withdrawn.

To decide the classes, we get the distribution of the differences between the permits and the capacities.
Keep only the positive ones and of those use the 5% to get a mild deviation, and 98% to get the extreme

In [ ]:
DEV_CLASSES = ['MILD', 'SEVERE', 'EXTREME']
WD_COLUMNS = [f"{c[0]}-{c[1]}" for c in itertools.product(LOCATIONS, DEV_CLASSES)]
groundw_dps['water_displacement-fine_amount'] = pd.DataFrame(columns=WD_COLUMNS, index=pd.DatetimeIndex([], name='timestamp'))
df = groundw_dps['water_displacement-fine_amount']
df.loc["2000-01-01"] = [5e5, 1e7, 1e8]
df.loc["2006-01-01"] = [52e4, 13e6, 2e8]
df.loc["2015-01-01"] = [52e4, 17e6, 2e8]

In [ ]:
# Calculate the classes thresholds
QT_MILD = 0.05
QT_SEVERE = 0.98
df = sources_sps['groundwater'].set_index('source_id')
# For closed sources that will open, assume participants use the maximum 130% of the permit
mask = df['capacity-nominal'].isna()
df.loc[mask, 'capacity-nominal'] = df.loc[mask, 'permit']*1.3/365
df['deviations'] = df[['capacity-nominal', 'permit']].apply(
    lambda row: max(row['capacity-nominal'] * 365 - row['permit'], 0), axis=1
)
positive_devs = df['deviations'][df['deviations'] > 0]
mild_dev_value = positive_devs.quantile(QT_MILD)
severe_dev_value = positive_devs.quantile(QT_SEVERE)
print(f"The select upper thresholds for a mild and severe deviation of the permit are {mild_dev_value} and {severe_dev_value}")

In [ ]:
groundw_dps.dump(path='..')
surfacew_dps.dump(path='..')
desalin_dps.dump(path='..')